# Perfilado de Calidad de Datos SISAV2

Este notebook documenta el estado de calidad de los datos exportados desde SISAV2.  
Reutiliza las funciones de `src.ingest` y `src.profiling` sin duplicar lógica.  

**Objetivo**: caracterizar los datos reales para fundamentar las reglas de limpieza de la Fase 2.

---

In [ ]:
import sys
from pathlib import Path

# Asegurar que src/ sea importable independientemente de dónde inicie el kernel.
# Cursor/VS Code inician el kernel en la raíz del proyecto; jupyter clásico en notebooks/.
_this_dir = Path("__file__").resolve().parent if "__file__" in dir() else Path.cwd()
_candidates = [_this_dir.parent, _this_dir, Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p for p in _candidates if (p / "src" / "__init__.py").exists()), _candidates[0])

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.ingest import leer_excels
from src.profiling import matriz_cobertura, resumen_categoricos, detectar_problemas
from src.schema import COLUMNAS_CANONICAS

sns.set_theme(style="whitegrid", font_scale=0.85)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 100)

## 1. Ingesta de datos

Se cargan todos los archivos `.xlsx` del directorio de exports reales.  
Los archivos que no conforman el esquema canónico de 36 columnas se reportan y excluyen.

In [ ]:
DATA_DIR = Path("/Users/ignacioramirez/Desktop/Proyecto VCM/drive-download-20260623T174348Z-3-001/iniciativas_a_excel/")

df, rechazados = leer_excels(DATA_DIR, strict=True)

print(f"Archivos conformes: {df['_archivo_origen'].nunique()}")
print(f"Filas consolidadas: {df.shape[0]}")
print(f"Archivos rechazados: {len(rechazados)}")
print()
for r in rechazados:
    print(f"  ✗ {r['archivo'][:75]}")

### Distribución de filas por convocatoria

In [ ]:
dist = (
    df.groupby(["_convocatoria", "_instrumento", "_anio"])
    .size()
    .reset_index(name="filas")
    .sort_values("_anio")
)
display(dist.style.hide(axis="index"))

### Verificación de metadatos parseados del filename

Los metadatos `_instrumento`, `_anio` y `_semestre` se extraen del nombre del archivo
mediante regex (ver `src.ingest.parsear_nombre_archivo`). No vienen como columnas en el Excel.

**Es crítico verificar que el parseo es correcto antes de confiar en cualquier análisis
temporal o de segmentación por instrumento.** Un regex que tome un número incorrecto
como año (e.g. un número de convocatoria) produciría sesgos silenciosos en el dashboard.

In [ ]:
from datetime import date

# Seleccionar columnas de linaje presentes en el DataFrame
_meta_cols = ["_archivo_origen", "_nivel", "_instrumento", "_convocatoria", "_anio", "_semestre"]
_meta_cols = [c for c in _meta_cols if c in df.columns]

meta = (
    df[_meta_cols]
    .drop_duplicates()
    .sort_values("_anio")
    .reset_index(drop=True)
)

display(meta.style.hide(axis="index"))

# --- Diagnóstico ---
n_anio_none = meta["_anio"].isna().sum()
n_sem_none = meta["_semestre"].isna().sum()
n_instr_desc = (meta["_instrumento"] == "DESCONOCIDO").sum() if "_instrumento" in meta.columns else 0

print(f"\nArchivos con _anio = None: {n_anio_none} de {len(meta)}")
print(f"Archivos con _semestre = None: {n_sem_none} de {len(meta)}")
print(f"Archivos con _instrumento = DESCONOCIDO: {n_instr_desc} de {len(meta)}")

# Alertar años implausibles: < 2015 o > año_actual + 5
ANIO_MIN = 2015
ANIO_MAX = date.today().year + 5
años_validos = meta["_anio"].dropna()
fuera_rango = meta[meta["_anio"].notna() & ((meta["_anio"] < ANIO_MIN) | (meta["_anio"] > ANIO_MAX))]
if not fuera_rango.empty:
    print(f"\n  ALERTA: {len(fuera_rango)} archivo(s) con año implausible (fuera de {ANIO_MIN}–{ANIO_MAX}):")
    display(fuera_rango.style.set_properties(**{"background-color": "#ffcdd2"}))
else:
    print(f"\n Todos los años están dentro del rango plausible ({ANIO_MIN}–{ANIO_MAX}).")

# Mostrar filas con _anio = None si las hay
if n_anio_none > 0:
    print(f"\n  Archivos sin año detectado:")
    display(meta[meta["_anio"].isna()].style.set_properties(**{"background-color": "#fff9c4"}))

---
## 2. Matriz de cobertura

Porcentaje de valores **no nulos** por columna y convocatoria.  
Permite identificar de un vistazo qué campos existen en qué años/instrumentos.

In [ ]:
cob = matriz_cobertura(df)

fig, ax = plt.subplots(figsize=(20, 12))
sns.heatmap(
    cob.T,
    cmap="YlOrRd",
    vmin=0,
    vmax=100,
    linewidths=0.3,
    annot=False,
    cbar_kws={"label": "% poblado"},
    ax=ax,
)
ax.set_title("Matriz de cobertura: % de valores no-nulos por columna × convocatoria", fontsize=13)
ax.set_xlabel("Convocatoria")
ax.set_ylabel("Columna")
plt.tight_layout()
plt.show()

### Resumen de cobertura global

In [ ]:
global_pct = df[COLUMNAS_CANONICAS].notna().mean() * 100

cobertura_df = pd.DataFrame({
    "columna": global_pct.index,
    "pct_poblado": global_pct.values.round(1),
}).sort_values("pct_poblado")

fig, ax = plt.subplots(figsize=(10, 10))
colors = ["#d32f2f" if v == 0 else "#ff9800" if v < 50 else "#4caf50" for v in cobertura_df["pct_poblado"]]
ax.barh(cobertura_df["columna"], cobertura_df["pct_poblado"], color=colors)
ax.set_xlabel("% poblado global")
ax.set_title("Cobertura global por columna")
ax.axvline(50, color="gray", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

print(f"\nColumnas 100% vacías ({(global_pct == 0).sum()}):")
for c in global_pct[global_pct == 0].index:
    print(f"  • {c}")

print(f"\nColumnas 100% pobladas ({(global_pct == 100).sum()}):")
for c in global_pct[global_pct == 100].index:
    print(f"  • {c}")

---
## 3. Campos categóricos: cardinalidad y valores frecuentes

Se analizan los campos con ≤30 valores únicos (candidatos a vocabulario controlado).

In [ ]:
cats = resumen_categoricos(df)

for col, info in sorted(cats.items(), key=lambda x: -x[1]["cardinalidad"]):
    if info["cardinalidad"] == 0:
        continue
    print(f"\n{'='*70}")
    print(f"{col}  (cardinalidad = {info['cardinalidad']})")
    print(f"{'='*70}")
    top_vals = pd.DataFrame(
        list(info["valores"].items())[:15],
        columns=["valor", "frecuencia"]
    )
    display(top_vals.style.hide(axis="index"))

---
## 4. Problemas de suciedad detectados

Cada subsección presenta un tipo de problema con evidencia concreta extraída de los datos.

In [ ]:
problemas = detectar_problemas(df)

### 4.1 Espacios y tabs (padding)

Valores con espacios o tabuladores al inicio/final. Rompe joins, agrupaciones y visualizaciones.

In [ ]:
padding_cols = []
for col in COLUMNAS_CANONICAS:
    serie = df[col].dropna().astype(str)
    con_padding = serie[serie != serie.str.strip()]
    if len(con_padding) > 0:
        padding_cols.append({"columna": col, "valores_con_padding": len(con_padding), "ejemplo": repr(con_padding.iloc[0][:60])})

if padding_cols:
    display(pd.DataFrame(padding_cols).style.hide(axis="index"))
else:
    print("No se detectó padding.")

### 4.2 Puntos finales en categorías

Valores categóricos que terminan en `.` - artefacto del formulario SISAV2 donde cada opción se almacena con punto final.

In [ ]:
puntos_cols = []
for col in COLUMNAS_CANONICAS:
    serie = df[col].dropna().astype(str)
    con_punto = serie[serie.str.endswith(".") & (serie.str.len() < 100)]
    if len(con_punto) > 0:
        ejemplos = con_punto.value_counts().head(3).index.tolist()
        puntos_cols.append({"columna": col, "n_valores": len(con_punto), "ejemplos": ejemplos})

if puntos_cols:
    display(pd.DataFrame(puntos_cols).style.hide(axis="index"))
else:
    print("No se detectaron puntos finales.")

### 4.3 Campos multivalor con separador

Campos que contienen múltiples valores concatenados con `"; "` o `" / "`.  
Implica que la columna no es atómica - cada celda tiene una lista serializada.

In [ ]:
import re

multi_cols = []
for col in COLUMNAS_CANONICAS:
    serie = df[col].dropna().astype(str)
    # Excluir fechas (DD/MM/YYYY) del análisis de separadores
    if col in ("Fecha Inicio", "Fecha Termino"):
        continue
    con_sep = serie[serie.str.contains(r";\ |\ /\ ", regex=True, na=False)]
    if len(con_sep) > 5:
        ejemplo = con_sep.iloc[0][:80]
        multi_cols.append({"columna": col, "n_multivalor": len(con_sep), "ejemplo": ejemplo})

if multi_cols:
    display(pd.DataFrame(multi_cols).sort_values("n_multivalor", ascending=False).style.hide(axis="index"))
else:
    print("No se detectaron campos multivalor.")

### 4.4 Semestre Ejecución: alta variabilidad

Este campo debería representar ~3 categorías (1er Semestre, 2do Semestre, Anual) pero contiene ~75 variantes incluyendo duraciones y fechas libres.

In [ ]:
sem = df["Semestre Ejecución"].dropna().value_counts()
print(f"Valores únicos: {len(sem)}")
print(f"\nTop 10 (cubren {sem.head(10).sum()} de {sem.sum()} valores):")
display(sem.head(10).to_frame("frecuencia").style)

print(f"\nEjemplos de variantes marginales (cola larga):")
display(sem.tail(15).to_frame("frecuencia").style)

### 4.5 Montos en $0

El 61% de los montos no-nulos son exactamente `0`. ¿Son montos reales o representan "sin presupuesto asignado"?

In [ ]:
montos = pd.to_numeric(df["Monto"], errors="coerce")
total_no_null = montos.notna().sum()
ceros = (montos == 0).sum()
positivos = (montos > 0).sum()

print(f"Monto no-nulo: {total_no_null}")
print(f"  = 0: {ceros} ({ceros/total_no_null*100:.1f}%)")
print(f"  > 0: {positivos} ({positivos/total_no_null*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].pie([ceros, positivos], labels=["$0", "> $0"], autopct="%1.0f%%", colors=["#ffcdd2", "#c8e6c9"])
axes[0].set_title("Distribución de Monto (= 0 vs > 0)")

montos_pos = montos[montos > 0]
axes[1].hist(montos_pos, bins=30, color="#4caf50", edgecolor="white")
axes[1].set_xlabel("Monto ($CLP)")
axes[1].set_ylabel("Frecuencia")
axes[1].set_title("Distribución de montos > $0")

plt.tight_layout()
plt.show()

### 4.6 Columnas 100% vacías

10 columnas del esquema canónico no tienen un solo valor en ninguna de las 33 convocatorias conformes.  
Son campos del formulario SISAV2 que nunca fueron habilitados o poblados.

In [ ]:
vacias = [c for c in COLUMNAS_CANONICAS if df[c].isna().all()]
print(f"Columnas 100% vacías ({len(vacias)}):")
for c in vacias:
    print(f"  • {c}")

### 4.7 Fechas como strings

Los campos `Fecha Inicio` y `Fecha Termino` vienen como texto en formato `DD/MM/YYYY`.  
Requieren conversión a `datetime` para cálculos de duración y filtros temporales.

In [ ]:
for col in ["Fecha Inicio", "Fecha Termino"]:
    serie = df[col].dropna()
    print(f"{col}: {len(serie)} valores no-nulos")
    print(f"  Tipo actual: {serie.dtype}")
    print(f"  Ejemplos: {serie.head(5).tolist()}")
    print()

---
## 5. Mapeo de hallazgos a reglas de limpieza candidatas

Cada problema detectado se vincula a una regla de limpieza propuesta para la Fase 2.

| Hallazgo | Regla | Qué hace |
|----------|-------|----------|
| Padding en textos (§4.1) | **R-001** | Strip + colapso de espacios múltiples |
| Tabs y newlines embebidos (§4.1) | **R-002** | Reemplazo de `\t` y `\n` por espacio simple |
| Puntos finales en multivalor (§4.2) | **R-003** | Elimina `.` final de cada ítem en campos separados por `"; "` / `" / "` |
| Fechas como string (§4.7) | **R-004** | Conversión `DD/MM/YYYY` → `datetime64` |
| Semestre Ejecución caótico (§4.4) | **R-005** | Normalización a vocabulario: `1S`, `2S`, `Anual`, `Otro` |
| Monto = 0 como null implícito (§4.5) | **R-006** | `0` → `NaN` |
| 10 columnas 100% vacías (§4.6) | **R-007** | Eliminación de columnas sin dato |
| PLATAFORMA constante (§3) | **R-008** | Eliminación de columna invariante |
| Nombres de iniciativa con punto final (§4.2) | **R-009** | Strip de `.` final en títulos cortos |

**Siguiente paso**: revisar y aprobar estas reglas antes de implementar `src/transform.py` (Fase 2).

---
## 6. Validación post-limpieza

Sección que ejecuta el pipeline completo (ingest → transform) sobre los mismos datos
y compara métricas antes/después para demostrar la efectividad de las reglas implementadas.

In [ ]:
from src.transform import aplicar_todas
import pandas as pd
import matplotlib.pyplot as plt

# Copia pre-limpieza para comparar
df_pre = df.copy()

# Ejecutar pipeline de transformación
audit_log: list[dict] = []
df_post = aplicar_todas(df.copy(), audit_log)
audit_df = pd.DataFrame(audit_log)

print(f"Total de modificaciones registradas: {len(audit_log)}")
print(f"Filas procesadas: {len(df_post)}")
print()

### 6.1 Resumen de modificaciones por regla

In [ ]:
if not audit_df.empty:
    resumen_reglas = audit_df["regla_id"].value_counts().sort_index().reset_index()
    resumen_reglas.columns = ["Regla", "Modificaciones"]
    print("Modificaciones por regla:")
    print(resumen_reglas.to_string(index=False))
    print()

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(resumen_reglas["Regla"], resumen_reglas["Modificaciones"], color="#2196F3")
    ax.set_xlabel("Nº de modificaciones")
    ax.set_title("Modificaciones registradas por regla de limpieza")
    plt.tight_layout()
    plt.show()
else:
    print("No se registraron modificaciones.")

### 6.2 Comparación antes/después: padding y puntos finales

In [ ]:
from src.schema import COLUMNAS_CANONICAS

# Columnas de texto (excluir numéricas y de fecha)
_COLS_NO_TEXTO = {"N", "Monto", "Fecha Inicio", "Fecha Termino"}

def contar_padding(frame):
    """Cuenta valores con espacios extra (padding al inicio/final o múltiples internos)."""
    count = 0
    for col in COLUMNAS_CANONICAS:
        if col in _COLS_NO_TEXTO or col not in frame.columns:
            continue
        series = frame[col].dropna()
        series_str = series[series.apply(lambda x: isinstance(x, str))]
        if series_str.empty:
            continue
        count += int((series_str != series_str.str.strip()).sum())
        count += int(series_str.str.contains(r"  +", regex=True, na=False).sum())
    return count

# Detección de puntos finales en campos multivalor
COLS_MULTIVALOR = ["ODS", "Competencia Sello", "Área de influencia", "Competencia genérica", "PDI"]

def contar_puntos_mv(frame):
    """Cuenta valores con puntos finales en campos multivalor."""
    count = 0
    for col in COLS_MULTIVALOR:
        if col not in frame.columns:
            continue
        series = frame[col].dropna()
        series_str = series[series.apply(lambda x: isinstance(x, str))]
        if series_str.empty:
            continue
        count += int(series_str.str.contains(r"\.$", regex=True, na=False).sum())
    return count

comparacion = pd.DataFrame({
    "Métrica": [
        "Valores con padding (espacios extra)",
        "Campos multivalor con puntos finales",
    ],
    "Antes": [contar_padding(df_pre), contar_puntos_mv(df_pre)],
    "Después": [contar_padding(df_post), contar_puntos_mv(df_post)],
})
comparacion["Reducción"] = comparacion["Antes"] - comparacion["Después"]
print("Comparación antes/después:")
print(comparacion.to_string(index=False))
display(comparacion)

### 6.3 Semestre Ejecución: valores normalizados y "Otro"

In [ ]:
print("Distribución post-limpieza de Semestre Ejecución:")
sem_post = df_post["Semestre Ejecución"].value_counts(dropna=False).reset_index()
sem_post.columns = ["Valor", "Frecuencia"]
print(sem_post.to_string(index=False))
print()

# Valores originales que cayeron en "Otro"
if not audit_df.empty:
    otros = audit_df[
        (audit_df["regla_id"] == "R-005") &
        (audit_df["valor_resultante"] == "Otro")
    ][["valor_original", "codigo_iniciativa", "archivo_origen"]]
    if len(otros) > 0:
        print(f"Valores que cayeron en 'Otro' ({len(otros)} registros):")
        print(otros.to_string(index=False))
    else:
        print("Ningún valor cayó en 'Otro'.")
else:
    print("No hay audit log.")

### 6.4 Resumen del audit log (primeras filas)

In [ ]:
if not audit_df.empty:
    print(f"Audit log: {len(audit_df)} registros totales")
    print()
    print("Primeros 20 registros:")
    print(audit_df.head(20).to_string(index=False))
    print()

    # Desglose por columna (top 10)
    print("Top 10 columnas más modificadas:")
    top_cols = audit_df["columna"].value_counts().head(10).reset_index()
    top_cols.columns = ["Columna", "Modificaciones"]
    print(top_cols.to_string(index=False))
else:
    print("Audit log vacío.")